# **COVID Dashboard**

## 1\. Contexto

Projeto de criação de um dashboard claro e de facil utilização contendo dados sobre a pandemia da COVID-19 no Brasil.

Esse notebook foi projetado para realizar toda a manipulação dos dados para a montagem do dashboard. **CASO O INTERESSE SEJA NO DASHBOARD, O LINK DE ACESSO ESTÁ NO FINAL DA PAGINA**

Para realizar o estudo sobre os dados da pandemia utilizamos as seguintes fontes de dados:



<h2>Dados gerais sobre a pandemia no Brasil<h2>

Para verificarmos dados como o número total de casos e mortes relacionado ao Covid-19 utilizamos a base de dados da universidade John Hopkins.

[Clique aqui para verificar os dados](https://github.com/CSSEGISandData/COVID-19/tree/master/csse_covid_19_data/csse_covid_19_daily_reports)


<h2>Dados sobre a vacinação do Brasil<h2>

Para verificarmos dados sobre todas as 3 levas de vacinação que ocorreram no Brasil, e o número total de população do brasil, utilizamos a base de dados de vacinação organização Our World in Data


[Clique aqui para verificar os dados](https://https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv)

<h2>Sobre a manipulação de Dados<h2>

Utilizamos os dados das fontes para realizarmos diversas operações matematicas para melhor representação do Dashboard. Por exemplo temos as porcentagens da vacinação, o calculo do número de novos casos e novas mortes por dia, e a media movel de casos e mortes.


## 2\. Pacotes e bibliotecas

In [1]:
import math
from typing import Iterator
from datetime import datetime, timedelta

import numpy as np
import pandas as pd


## 3\. Extração

### 3.1 Casos Covid

In [2]:
def date_range(data_inicial: datetime, data_final: datetime) -> Iterator[datetime]:
  intervalo_de_dias: int = (data_final - data_inicial).days
  for dia in range(intervalo_de_dias + 1):
    yield data_inicial + timedelta(dia)

dados_covid = []

DATA_INICIAL = datetime(2021, 1, 1)
DATA_FINAL = datetime(2023, 3, 9)
COLUNAS_DE_INTERESSE = ['Province_State', 'Country_Region', 'Confirmed', 'Deaths', 'Incident_Rate']


for data in pd.date_range(DATA_INICIAL, DATA_FINAL, freq='D'):
  data_str = data.strftime('%m-%d-%Y')
  url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_daily_reports/{data_str}.csv'

  try:
    caso = pd.read_csv(url, sep=',')
  except Exception as e:
    print(f'Erro em {data_str}: {e}')
    continue

  caso = caso[COLUNAS_DE_INTERESSE]
  caso = caso.query('Country_Region == "Brazil"').reset_index(drop=True)
  caso['Data'] = pd.to_datetime(data.strftime('%Y%m%d'))

  dados_covid.append(caso)

casos_covid = pd.concat(dados_covid, ignore_index=True)

### 3.2 Dados vacinação

In [3]:
URL_VACINAS = 'https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv'

vacinas = pd.read_csv(URL_VACINAS, sep=',', parse_dates=[3])

/tmp/ipykernel_13482/3919322347.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  vacinas = pd.read_csv(URL_VACINAS, sep=',', parse_dates=[3])


## 4\. Transformação

### 4.1 Casos Covid

In [4]:
colunas_tendencia = ['tendencia_de_casos', 'tendencia_de_mortes']
TENDENCIA_MENOR = 0.75
TENDENCIA_MAIOR = 1.15
NOMES_ESTADOS = {
    'Amapa': 'Amapá',
    'Ceara': 'Ceará',
    'Espirito Santo': 'Espírito Santo',
    'Goias': 'Goiás',
    'Para': 'Pará',
    'Paraiba': 'Paraíba',
    'Parana': 'Paraná',
    'Piaui': 'Piauí',
    'Rondonia': 'Rondônia',
    'Sao Paulo': 'São Paulo'
}

COLUNAS = {
    'Province_State': 'estado',
    'Country_Region': 'pais',
    'Confirmed': 'casos',
    'Deaths': 'mortes',
    'Incident_Rate': 'taxa_de_infeccao',
    'Data' : 'data'
}

def obter_tendencia(taxa: float) -> str:
  if np.isnan(taxa):
    return np.nan
  if taxa < TENDENCIA_MENOR:
    return 'Abaixando'
  if taxa > TENDENCIA_MAIOR:
    return 'Aumentando'
  else:
    return 'Estabilizado'

def ajustar_nomes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=COLUNAS)
    df['estado'] = df['estado'].map(NOMES_ESTADOS).fillna(df['estado'])
    return df

def computar_metricas(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
  novos_casos = f'{coluna}_em_24h'
  media_7_dias = f'{coluna}_media_7d'
  media_movel = f'{coluna}_media_movel'
  tendencia = f'tendencia_de_{coluna}'

  df[novos_casos] = df[coluna].diff(periods=1)
  df[media_7_dias] = np.ceil(df[novos_casos].rolling(window=7).mean())
  df[media_movel] = df[media_7_dias] / df[media_7_dias].shift(periods=14)
  df[tendencia] = df[media_movel].apply(obter_tendencia)
  return df

def processamento_estados(df: pd.DataFrame) -> pd.DataFrame:
  df = ajustar_nomes(df)
  df = df.sort_values('data').reset_index(drop=True)
  df = computar_metricas(df, 'casos')
  df = computar_metricas(df, 'mortes')

  return df


casos_covid = (casos_covid.groupby('Province_State', group_keys=False)).apply(processamento_estados).reset_index(drop=True)
casos_covid[colunas_tendencia] = casos_covid[colunas_tendencia].fillna('N/A')


/tmp/ipykernel_13482/1904863492.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  casos_covid = (casos_covid.groupby('Province_State', group_keys=False)).apply(processamento_estados).reset_index(drop=True)


### 4.2 Vacinas

In [5]:
COLUNAS_VACINAS = ['country', 'population', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters', 'date']
NOMES_AJUSTADOS_VACINA={
    'date': 'data',
    'total_vaccinations': 'total_de_vacinados',
    'people_vaccinated': 'primeira_dose',
    'people_fully_vaccinated': 'segunda_dose',
    'total_boosters': 'dose_reforco'
}



def calcular_porcentagem(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
  df[f'porcentagem_{coluna}'] = round(df[coluna] / df['population'], 4)
  return df


def filtrar_datas(df: pd.DataFrame, data_inicio: str, data_fim: str) -> pd.DataFrame:
    df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')
    return df.loc[df['data'].between(data_inicio, data_fim)].reset_index(drop=True)


vacinas = vacinas.query('country == "Brazil"').reset_index(drop=True)
vacinas = vacinas[COLUNAS_VACINAS]
vacinas = vacinas.rename(columns= NOMES_AJUSTADOS_VACINA)
vacinas = filtrar_datas(vacinas, DATA_INICIAL, DATA_FINAL)
vacinas = calcular_porcentagem(vacinas, 'primeira_dose')
vacinas = calcular_porcentagem(vacinas, 'segunda_dose')
vacinas = calcular_porcentagem(vacinas, 'dose_reforco')
vacinas = vacinas.fillna(method='ffill')

/tmp/ipykernel_13482/1556663374.py:29: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  vacinas = vacinas.fillna(method='ffill')


### 4.3 Dados combinados

Devido a uma limitação na combinação de dados de duas fontes diferentes pelo Google Data Studio, observei a necessidade de criar uma terceira fonte de dados para facilitar a criação do grafico contendo uma relação direta entre os numeros de novos casos, mortes e as levas de vacinações.

In [6]:
COLUNAS_VAC_COMBINADAS = ['data', 'porcentagem_primeira_dose', 'porcentagem_segunda_dose', 'porcentagem_dose_reforco']

casos_por_dia = (casos_covid.groupby('data', as_index=False)).agg(
    novos_casos = ('casos_em_24h', 'sum'),
    novas_mortes = ('mortes_em_24h', 'sum'),
    total_casos = ('casos', 'sum'),
    total_mortes = ('mortes', 'sum')
)


dados_grafico = pd.merge(casos_por_dia, vacinas[COLUNAS_VAC_COMBINADAS], on='data', how='left').sort_values('data').reset_index(drop=True)

## 5\. Carregamento

Arquivos que foram utilizados para a construção das fontes de dados do Google Data Studio.

In [7]:
vacinas.to_csv('listagem_vacinas.csv', index=False)
casos_covid.to_csv('casos_covid.csv', index=False)
dados_grafico.to_csv('dados_grafico_covid19.csv', index=False)

## 6. Publicação

Essa atividade foi projetada como uma etapa do curso de analise de dados da EBAC, onde visava a manipulação de dados e montagem de um dashboard no Google Data Studio.

<h1>Links Importantes<h1>

* Dashboard: [Clique aqui](https://datastudio.google.com/reporting/9af92fcc-a569-45f3-b13e-56e28f31ce1f/page/51PzF?s=iSB5KYLuyVk)
* Notebook Google Colab (RECOMENDADO): [Clique aqui](https://colab.research.google.com/drive/1pSoAo29OHC2_szT1x6OcIjxtnyIAFLi5?usp=sharing)
* Notebook Kagle: [Clique aqui](https://www.kaggle.com/code/roneypasolini/projeto-analise-de-dados-covid-19-no-brasil)
* Github: [Clique aqui](https://github.com/RoneyPasolini/Projeto-Analise-de-Dados-Covid-19-no-Brasil)

<h2>Sobre mim<h2>

Me chamo Roney Pasolini Souza Castro, sou formado em ciência da computação pela Faesa. Com muito interesse em migrar o meu trabalho para a area de dados.
Caso haja interesse em minhas habilidades, ou queria reportar uma melhoria ao meu trabalho, estarei disponivel para contato via e-mail -> roney.pasolini.contato@gmail.com <-

Agradeço desde já a atenção.
Roney Pasolini Souza Castro